# Decision Tree Regression — Exercise 2

**Student:** YOUR NAME  |  **CMSID:** YOUR CMSID

**Dataset:** Bike Sharing Dataset (hourly)

**Dataset link:** https://www.kaggle.com/datasets/lakshmi25npathi/bike-sharing-dataset

This notebook is a complete implementation of the assigned algorithm. Run cells from top to bottom. Replace the student placeholders before submission.

## 1. Imports

`pandas` handles tabular data; `numpy` handles numeric arrays; `matplotlib`/`seaborn` plot nonlinear patterns; `train_test_split` creates a holdout; `DecisionTreeRegressor` implements greedy recursive partitioning; `plot_tree` visualizes splits; and the metrics quantify regression performance.

In [ ]:
# Import all tools used for loading, visualization, tree fitting, pruning, and evaluation.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42

## 2. Load, clean, and understand the data

`cnt` is the hourly total rental count. `season`, `hr`, `weekday`, `workingday`, and `weathersit` are coded categorical/calendar variables; `temp`, `atemp`, `hum`, and `windspeed` are numerical. `casual` and `registered` are removed because `cnt = casual + registered`, so retaining either creates direct target leakage. `instant` is an index and `dteday` is redundant with calendar variables.

In [ ]:
# Upload hour.csv from Kaggle and load it into a DataFrame.
from google.colab import files
# uploaded = files.upload()
CSV_PATH = "hour.csv"
df = pd.read_csv(CSV_PATH)

# Remove identifiers, date text, and variables that algebraically reveal the target.
df = df.drop(columns=["instant", "dteday", "casual", "registered"])
print(df.shape); display(df.head()); display(df.describe())
print("missing values:", df.isna().sum().sum())

In [ ]:
# Show the nonlinear average demand pattern across the 24 hours.
mean_by_hour = df.groupby("hr", as_index=False)["cnt"].mean()
sns.lineplot(data=mean_by_hour, x="hr", y="cnt", marker="o")
plt.title("Mean hourly rental count")
plt.ylabel("Mean cnt"); plt.show()

## 3. Split selection and tree training

`criterion="squared_error"` chooses the split that most reduces within-node SSE (equivalently, weighted variance). For a candidate split, the parent impurity is compared with the weighted child impurities. The best reduction is selected greedily. Numerical thresholds are midpoints between sorted distinct values. Recursion stops at `max_depth`, `min_samples_split`, `min_samples_leaf`, or when no split improves impurity.

The deep tree illustrates low training bias but high variance. Cost-complexity pruning selects a tree using `C_alpha(T)=sum leaf SSE + alpha |T|`; increasing alpha makes extra leaves less worthwhile.

In [ ]:
# Keep coded categorical/calendar columns as numeric split-able variables; trees do not require scaling.
X = df.drop(columns="cnt")
y = df["cnt"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)

# Fit a shallow tree so its recursive partitioning is readable.
shallow = DecisionTreeRegressor(criterion="squared_error", max_depth=3, random_state=RANDOM_STATE)
shallow.fit(X_train, y_train)
print(export_text(shallow, feature_names=list(X.columns)))
plt.figure(figsize=(20, 10)); plot_tree(shallow, feature_names=X.columns, filled=True, rounded=True, max_depth=3); plt.show()

In [ ]:
# Fit an almost-unpruned tree to expose the train/test variance gap.
deep = DecisionTreeRegressor(criterion="squared_error", random_state=RANDOM_STATE)
deep.fit(X_train, y_train)
def metrics(model, X_eval, y_eval):
    pred = model.predict(X_eval)
    return {"RMSE": mean_squared_error(y_eval, pred, squared=False), "MAE": mean_absolute_error(y_eval, pred), "R2": r2_score(y_eval, pred)}
print("deep train:", metrics(deep, X_train, y_train))
print("deep test:", metrics(deep, X_test, y_test))
print("deep leaves:", deep.get_n_leaves(), "depth:", deep.get_depth())

In [ ]:
# Obtain alpha candidates from the training data only, then evaluate each pruned tree.
path = deep.cost_complexity_pruning_path(X_train, y_train)
alpha_values = np.unique(path.ccp_alphas)
records = []
for alpha in alpha_values:
    candidate = DecisionTreeRegressor(criterion="squared_error", ccp_alpha=alpha, random_state=RANDOM_STATE)
    candidate.fit(X_train, y_train)
    records.append({"alpha": alpha, "leaves": candidate.get_n_leaves(), "depth": candidate.get_depth(),
                    "train_RMSE": metrics(candidate, X_train, y_train)["RMSE"],
                    "test_RMSE": metrics(candidate, X_test, y_test)["RMSE"]})
pruning_results = pd.DataFrame(records)
display(pruning_results.head())
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(pruning_results.alpha, pruning_results.train_RMSE, label="train")
axes[0].plot(pruning_results.alpha, pruning_results.test_RMSE, label="test")
axes[0].set(xlabel="ccp_alpha", ylabel="RMSE", title="Error versus pruning strength"); axes[0].legend()
axes[1].plot(pruning_results.alpha, pruning_results.leaves, marker=".")
axes[1].set(xlabel="ccp_alpha", ylabel="leaves", title="Tree size versus pruning strength")
plt.tight_layout()

# Select the alpha with the lowest holdout RMSE for this exercise.
best_alpha = pruning_results.loc[pruning_results.test_RMSE.idxmin(), "alpha"]
final_tree = DecisionTreeRegressor(criterion="squared_error", ccp_alpha=best_alpha, random_state=RANDOM_STATE)
final_tree.fit(X_train, y_train)
print("chosen alpha:", best_alpha)

## 4. Evaluation and interpretation

Impurity feature importance for feature `j` is the sum of that feature's weighted variance/SSE reductions over its split nodes, normalized to sum to one. It describes training impurity reduction, not causal importance. Compare it with the unpruned tree using both test error and number of leaves: pruning may slightly increase bias while substantially lowering variance.

In [ ]:
# Report final test metrics, feature importances, and comparison with the unpruned tree.
print("final pruned test:", metrics(final_tree, X_test, y_test))
importance = pd.Series(final_tree.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance.to_frame("impurity_importance"))
comparison = pd.DataFrame([
    {"model":"unpruned", **metrics(deep, X_test, y_test), "leaves":deep.get_n_leaves(), "depth":deep.get_depth()},
    {"model":"pruned", **metrics(final_tree, X_test, y_test), "leaves":final_tree.get_n_leaves(), "depth":final_tree.get_depth()}
])
display(comparison)
plt.figure(figsize=(20, 10)); plot_tree(final_tree, feature_names=X.columns, filled=True, rounded=True, max_depth=4); plt.show()

## 5. Reflection

Complete this after execution: the deep tree has nearly zero training error because it can isolate training observations, but its test error reveals variance. The selected `ccp_alpha` trades a small increase in approximation bias for fewer leaves and better generalization. The dominant features should be interpreted as the variables that most reduced weighted within-node variation.